# Notebook 12 — MMR Diversity Re-Ranking

**Goal:** Re-rank Notebook 10 DeepFM Top-100 candidates with Maximal Marginal Relevance (MMR). Tune λ **only on August user-disjoint holdout**; evaluate September once. This is an offline diversity experiment, not an ad CTR experiment.

**Inputs:** Notebook 10 scored candidate CSVs, Notebook 06 split-specific Item2Vec models, Notebook 02 warm-user labels. No future labels are used for MMR scoring. **Output:** `reports/notebook12_mmr_reranking.json`, scored CSVs, metrics.

**Important:** Embedding coverage is partial; missing vectors have similarity zero and are explicitly tracked. The MMR diversity proxy is based on historical co-view embeddings, not product taxonomy or actual ad creatives.

## 0. Setup
Run from project root or `notebooks/`. Dependencies: `pip install pandas numpy gensim scipy scikit-learn`.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split

ROOT=Path.cwd().resolve()
if not (ROOT/'data'/'processed').is_dir(): ROOT=ROOT.parent
assert (ROOT/'data'/'processed').is_dir(), 'Run from project root or notebooks/'
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; MODELS=ROOT/'models'
REPORTS.mkdir(exist_ok=True)
SEED=42; K_VALUES=(10,20,50,100); LAMBDAS=(1.0,0.9,0.8,0.7,0.5)
TOP_K=10
raw={}; vectors={}
for split,stem in [('val','validation'),('test','test')]:
    path=PROC/f"notebook10_{split}_{'holdout_' if split=='val' else ''}scored_candidates.csv"
    assert path.exists(),f'Missing {path}: run Notebook 10 first'
    df=pd.read_csv(path)
    required={'visitorid','itemid','rank','score','new_rank'}
    assert required.issubset(df.columns), (path,required-set(df.columns))
    assert not df[['visitorid','itemid']].duplicated().any()
    assert df.groupby('visitorid').size().eq(100).all()
    assert np.isfinite(df.score).all()
    labels=pd.read_csv(PROC/f'{stem}_view_labels_warm_users.csv',usecols=['visitorid','itemid']).drop_duplicates()
    if split=='val':
        all_val_users=np.sort(pd.read_csv(PROC/'notebook08_val_hybrid_candidates.csv',usecols=['visitorid']).visitorid.unique())
        _,holdout_users=train_test_split(all_val_users,test_size=.20,random_state=SEED)
        assert set(df.visitorid.unique())==set(holdout_users),'August holdout cohort mismatch'
        labels=labels.loc[labels.visitorid.isin(holdout_users)].copy()
    assert set(df.visitorid.unique())==set(labels.visitorid.unique())
    model_path=MODELS/f'notebook06_{split}_item2vec.model'
    assert model_path.exists(),f'Missing {model_path}: run Notebook 06 first'
    vectors[split]=Word2Vec.load(str(model_path)).wv
    raw[split]=(df,labels)
    print(split,'users:',df.visitorid.nunique(),'candidates:',len(df),'vectors:',len(vectors[split]))

val users: 2666 candidates: 266600 vectors: 43267
test users: 15137 candidates: 1513700 vectors: 52280


## 1. MMR algorithm

At each step, choose the candidate maximizing `λ × normalized_relevance − (1−λ) × max_similarity_to_selected`. DeepFM scores are **per-user min-max normalized**; cosine similarity is clipped to `[0,1]` to penalize redundancy rather than reward negative similarity. Tie-break by original DeepFM rank then item ID. Missing embeddings receive zero similarity; this is a limitation, not evidence of true diversity. Keep the remaining 90 items in their original DeepFM order so the complete Top-100 candidate pool is unchanged.

In [2]:
def user_vectors(items,wv):
    mat=np.zeros((len(items),wv.vector_size),dtype=np.float32)
    known=np.zeros(len(items),dtype=bool)
    for j,item in enumerate(items):
        key=str(int(item))
        if key in wv:
            mat[j]=wv[key]; known[j]=True
    norms=np.linalg.norm(mat,axis=1,keepdims=True)
    np.divide(mat,np.maximum(norms,1e-12),out=mat)
    return mat,known

def mmr_user(items,scores,wv,lam,top_k=TOP_K):
    items=np.asarray(items,dtype=np.int64)
    scores=np.asarray(scores,dtype=np.float64)
    n=len(items); assert n>=top_k
    # caller passes items in DeepFM rank order; this index provides stable tie-breaking
    lo,hi=float(scores.min()),float(scores.max())
    relevance=(scores-lo)/(hi-lo) if hi>lo else np.zeros(n,dtype=np.float64)
    mat,known=user_vectors(items,wv)
    sim=np.clip(mat@mat.T,0,1)
    np.fill_diagonal(sim,0)
    selected=[];available=np.ones(n,dtype=bool)
    redundancy=np.zeros(n,dtype=np.float64)
    for _ in range(top_k):
        mmr=lam*relevance-(1-lam)*redundancy
        mmr[~available]=-np.inf
        choice=int(np.argmax(mmr)) # ties use DeepFM order
        selected.append(choice);available[choice]=False
        redundancy=np.maximum(redundancy,sim[:,choice])
    order=np.r_[np.array(selected,dtype=int),np.flatnonzero(available)]
    # cosine pairwise similarity among selected items where both embeddings exist
    selected_known=np.array(selected,dtype=int)[known[selected]]
    if len(selected_known)>=2:
        sub=sim[np.ix_(selected_known,selected_known)]
        tri=sub[np.triu_indices(len(selected_known),1)]
        pair_sim=float(tri.mean())
        pairs=len(tri)
    else:pair_sim=np.nan;pairs=0
    return order,{'embedded_top10':int(known[selected].sum()),'pairwise_similarity':pair_sim,'pair_count':pairs}

def apply_mmr(frame,wv,lam):
    rows=[];stats=[]
    ordered=frame.sort_values(['visitorid','new_rank','itemid'],kind='mergesort')
    for u,g in ordered.groupby('visitorid',sort=False):
        items=g.itemid.to_numpy();scores=g.score.to_numpy()
        idx,st=mmr_user(items,scores,wv,lam)
        out=g.iloc[idx][['visitorid','itemid','rank','score','new_rank']].copy()
        out['mmr_rank']=np.arange(1,len(out)+1,dtype=np.int16)
        rows.append(out);stats.append(st)
    return pd.concat(rows,ignore_index=True),pd.DataFrame(stats)

# Basic invariants / unit tests
class DummyVectors:
    vector_size=2
    def __contains__(self,k):return k in {'1','2','3'}
    def __getitem__(self,k):return {'1':np.array([1.,0.]),'2':np.array([1.,0.]),'3':np.array([0.,1.])}[k]
test_items=[1,2,3,4];test_scores=[.9,.8,.7,.6]
assert mmr_user(test_items,test_scores,DummyVectors(),1.0,3)[0].tolist()==[0,1,2,3]
assert mmr_user(test_items,test_scores,DummyVectors(),0.5,3)[0][0]==0
print('PASS: MMR baseline λ=1 preserves DeepFM rank and basic invariants')

PASS: MMR baseline λ=1 preserves DeepFM rank and basic invariants


## 2. Evaluation metrics
Micro Recall denominator includes **all** warm-user future-view labels, even previously seen/cold items. Catalog Coverage uses the split-specific history item catalog. Intra-List Diversity (ILD) = `1 − mean clipped cosine similarity`, computed only on Top-10 pairs with both embeddings. Report pair coverage to prevent interpreting missing vectors as diversity.

In [3]:
def evaluate(ranked,labels,history_catalog,k_values=K_VALUES):
    truth=labels.groupby('visitorid').itemid.agg(lambda x:set(map(int,x))).to_dict()
    ranked=ranked.sort_values(['visitorid','mmr_rank'],kind='mergesort')
    recs=ranked.groupby('visitorid',sort=False).itemid.agg(list).to_dict()
    assert set(recs)==set(truth)
    denom=sum(map(len,truth.values()))
    available_hits=sum(len(set(items)&truth[int(u)]) for u,items in recs.items())
    out=[]
    for k in k_values:
        hits=0;hit_users=0;macro=[];recommended=set()
        for u,items in recs.items():
            top=items[:k]; h=len(set(top)&truth[int(u)])
            hits+=h;hit_users+=int(h>0);macro.append(h/len(truth[int(u)]))
            recommended.update(top)
        out.append({'K':k,'Recall_micro':hits/denom,'Recall_macro':float(np.mean(macro)),
                    'HitRate':hit_users/len(truth),'CandidateConditionedRecall':hits/available_hits if available_hits else None,
                    'CatalogCoverage':len(recommended&history_catalog)/len(history_catalog),
                    'hits':hits})
    return pd.DataFrame(out)

def diversity_summary(stats):
    pairs=int(stats.pair_count.sum())
    weighted_sim=float(np.nansum(stats.pairwise_similarity*stats.pair_count)/pairs) if pairs else None
    # Pair coverage relative to 45 possible pairs per 10-item list
    return {'IntraListDiversity_known_pairs':1-weighted_sim if weighted_sim is not None else None,
            'known_pair_fraction':pairs/(45*len(stats)),
            'mean_embedded_items_at_10':float(stats.embedded_top10.mean())}

history_catalog={}
for split,filename in [('val','train_history.csv'),('test','test_history.csv')]:
    history_catalog[split]=set(pd.read_csv(PROC/filename,usecols=['itemid']).itemid.unique())
    print(split,'historical catalog:',len(history_catalog[split]))

val historical catalog: 199689
test historical catalog: 222982


## 3. Select λ using August holdout ONLY
Primary objective: maximize Recall@10 subject to a small relevance-loss budget (at most **1% relative** below λ=1) and then maximize diversity. This avoids picking a diversity-heavy configuration with unacceptable relevance loss. The budget and λ grid are fixed before inspecting September.

In [4]:
t0=time.perf_counter()
val_frame,val_labels=raw['val']
val_outputs={};val_stats={};holdout_rows=[]
for lam in LAMBDAS:
    ranked,stats=apply_mmr(val_frame,vectors['val'],lam)
    metrics=evaluate(ranked,val_labels,history_catalog['val'])
    div=diversity_summary(stats)
    row={'lambda':lam,**div,**{f'Recall_at_{int(r.K)}':float(r.Recall_micro) for r in metrics.itertuples()},
         'HitRate_at_10':float(metrics.loc[metrics.K.eq(10),'HitRate'].iloc[0]),
         'Coverage_at_10':float(metrics.loc[metrics.K.eq(10),'CatalogCoverage'].iloc[0])}
    holdout_rows.append(row);val_outputs[lam]=ranked;val_stats[lam]=stats
    print('lambda',lam,'holdout R@10',round(row['Recall_at_10'],6),'ILD',div['IntraListDiversity_known_pairs'])
holdout_table=pd.DataFrame(holdout_rows)
base_recall=float(holdout_table.loc[holdout_table['lambda'].eq(1.0),'Recall_at_10'].iloc[0])
eligible=holdout_table.loc[holdout_table.Recall_at_10>=base_recall*.99].copy()
assert len(eligible)>0
# Higher diversity first; tie-break higher Recall@10, then larger lambda.
eligible['selection_ild']=eligible.IntraListDiversity_known_pairs.fillna(-1)
selected=eligible.sort_values(['selection_ild','Recall_at_10','lambda'],ascending=[False,False,False],kind='mergesort').iloc[0]
SELECTED_LAMBDA=float(selected['lambda'])
print('Selected lambda on August holdout:',SELECTED_LAMBDA,'(Recall@10 floor:',base_recall*.99,')')
print('Holdout runtime seconds:',round(time.perf_counter()-t0,2))
display(holdout_table)

lambda 1.0 holdout R@10 0.034191 ILD 0.26110310211707766
lambda 0.9 holdout R@10 0.034383 ILD 0.2740472048032615
lambda 0.8 holdout R@10 0.033615 ILD 0.29700398204912737
lambda 0.7 holdout R@10 0.029005 ILD 0.35644116363332
lambda 0.5 holdout R@10 0.014022 ILD 0.5145153490450425
Selected lambda on August holdout: 0.9 (Recall@10 floor: 0.03384940453323088 )
Holdout runtime seconds: 43.45


,lambda,IntraListDiversity_known_pairs,known_pair_fraction,mean_embedded_items_at_10,Recall_at_10,Recall_at_20,Recall_at_50,Recall_at_100,HitRate_at_10,Coverage_at_10
0,1.0,0.261103,0.953913,9.747187,0.034191,0.045524,0.062044,0.072224,0.057389,0.060083
1,0.9,0.274047,0.902767,9.441110,0.034383,0.045716,0.062044,0.072224,0.058140,0.061245
2,0.8,0.297004,0.787489,8.719805,0.033615,0.045524,0.062044,0.072224,0.057014,0.064240
3,0.7,0.356441,0.561099,7.113278,0.029005,0.043604,0.062236,0.072224,0.051013,0.068942
4,0.5,0.514515,0.312862,4.402476,0.014022,0.039570,0.061083,0.072224,0.025506,0.076464


## 4. September test ONCE
Apply the already selected λ. No tuning or reselection using September labels.

In [5]:
test_frame,test_labels=raw['test']
t0=time.perf_counter()
test_baseline,test_base_stats=apply_mmr(test_frame,vectors['test'],1.0)
test_selected,test_selected_stats=apply_mmr(test_frame,vectors['test'],SELECTED_LAMBDA)
test_seconds=time.perf_counter()-t0
base_test=evaluate(test_baseline,test_labels,history_catalog['test'])
selected_test=evaluate(test_selected,test_labels,history_catalog['test'])
comparison=base_test.merge(selected_test,on='K',suffixes=('_deepfm','_mmr'))
print('Test baseline diversity:',diversity_summary(test_base_stats))
print('Test selected diversity:',diversity_summary(test_selected_stats))
print('Test batch re-ranking seconds (both runs, NOT p95):',round(test_seconds,2))
display(comparison)
assert np.array_equal(test_baseline.itemid.to_numpy(),test_frame.sort_values(['visitorid','new_rank','itemid'],kind='mergesort').itemid.to_numpy()),'lambda=1 must preserve DeepFM ranking'
assert (comparison.loc[comparison.K.eq(100),'Recall_micro_deepfm'].to_numpy()==comparison.loc[comparison.K.eq(100),'Recall_micro_mmr'].to_numpy()).all()
assert test_selected.groupby('visitorid').size().eq(100).all()
assert not test_selected[['visitorid','itemid']].duplicated().any()
print('PASS: baseline, fixed candidate pool, test protocol')

Test baseline diversity: {'IntraListDiversity_known_pairs': 0.2534975768221843, 'known_pair_fraction': 0.938581694596757, 'mean_embedded_items_at_10': 9.66202021536632}
Test selected diversity: {'IntraListDiversity_known_pairs': 0.2647606043048023, 'known_pair_fraction': 0.8744445178480985, 'mean_embedded_items_at_10': 9.276673052784568}
Test batch re-ranking seconds (both runs, NOT p95): 36.03


,K,Recall_micro_deepfm,Recall_macro_deepfm,HitRate_deepfm,CandidateConditionedRecall_deepfm,CatalogCoverage_deepfm,hits_deepfm,Recall_micro_mmr,Recall_macro_mmr,HitRate_mmr,CandidateConditionedRecall_mmr,CatalogCoverage_mmr,hits_mmr
0,10,0.027105,0.030666,0.048490,0.472389,0.145178,864,0.027011,0.030669,0.048557,0.470749,0.155936,861
1,20,0.037740,0.042260,0.064346,0.657736,0.214268,1203,0.037740,0.042290,0.064412,0.657736,0.215869,1203
2,50,0.050540,0.055258,0.080729,0.880809,0.319730,1611,0.050540,0.055258,0.080729,0.880809,0.319730,1611
3,100,0.057379,0.061646,0.088855,1.000000,0.438331,1829,0.057379,0.061646,0.088855,1.000000,0.438331,1829


PASS: baseline, fixed candidate pool, test protocol


## 5. Export reproducible artifacts

In [6]:
def records(df):return json.loads(df.to_json(orient='records'))
val_outputs[SELECTED_LAMBDA].to_csv(PROC/'notebook12_val_holdout_mmr_candidates.csv',index=False)
test_selected.to_csv(PROC/'notebook12_test_mmr_candidates.csv',index=False)
holdout_table.to_csv(REPORTS/'notebook12_val_lambda_sweep.csv',index=False)
comparison.to_csv(REPORTS/'notebook12_test_comparison.csv',index=False)
report={
 'method':'DeepFM MMR diversity re-ranking using split-specific Item2Vec cosine similarity',
 'protocol':'August holdout lambda selection; September test once; fixed Notebook08 Top-100 pool',
 'lambda_grid':list(LAMBDAS),'selected_lambda':SELECTED_LAMBDA,
 'selection_rule':'maximize known-pair ILD subject to holdout Recall@10 >= 99% of DeepFM baseline',
 'holdout_lambda_sweep':records(holdout_table),
 'test_metrics':records(comparison),
 'test_deepfm_diversity':diversity_summary(test_base_stats),
 'test_mmr_diversity':diversity_summary(test_selected_stats),
 'test_batch_seconds_two_runs':round(test_seconds,4),
 'caveats':['No true ad impressions or calibrated CTR/CVR','Implicit view labels only',
            'Item2Vec vectors missing for some items; ILD calculated only on known pairs',
            'Test history includes data prior to September cutoff; test Item2Vec is separately trained before September',
            'No production p95 latency measured','MMR re-ranks only Top-10; remaining Top-100 retains DeepFM order']}
path=REPORTS/'notebook12_mmr_reranking.json'
path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Saved:',path)
print('Next: Notebook 13 — cascaded Retrieval → LightGBM → DeepFM → MMR')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook12_mmr_reranking.json
Next: Notebook 13 — cascaded Retrieval → LightGBM → DeepFM → MMR


## Interpretation checklist

- Compare **August holdout** λ choices before looking at September test.
- If λ=1 is selected, the data did not support a diversity trade-off within the 1% Recall@10 budget; that is a valid result.
- Check `known_pair_fraction` before interpreting ILD; missing vectors are not automatically diverse.
- MMR scores are not probabilities. This notebook is a **standalone DeepFM→MMR** experiment; the true LightGBM→DeepFM cascade is built in Notebook 13.